# 📦 Phase 4 — Inventory Optimization
**Project**: Blinkit Supply Chain & Inventory Optimization  
**Author**: Rishi Verma

---
### 🎯 Goals:
- **ABC Analysis** — Classify products by revenue contribution (A=top 70%, B=next 20%, C=bottom 10%)
- **EOQ (Economic Order Quantity)** — Optimal order quantity per product
- **Reorder Point (ROP)** — When to trigger restocking
- **Damaged Stock Wastage Analysis** — Products with highest loss
- **Perishability Risk Scoring** — Expiry risk for short shelf-life items
- **Safety Stock Calculation** — Buffer against demand uncertainty
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import warnings, os

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

CLEANED = '../data/cleaned'
CHARTS  = '../outputs/charts'
REPORTS = '../outputs/reports'
os.makedirs(REPORTS, exist_ok=True)

products   = pd.read_csv(f'{CLEANED}/products_cleaned.csv')
inventory  = pd.read_csv(f'{CLEANED}/inventory_cleaned.csv', parse_dates=['date'])
order_items= pd.read_csv(f'{CLEANED}/order_items_cleaned.csv')
orders     = pd.read_csv(f'{CLEANED}/orders_cleaned.csv',   parse_dates=['order_date'])

print('✅ Data loaded!')
print(f'Products: {products.shape} | Inventory: {inventory.shape} | Order Items: {order_items.shape}')

## 1️⃣ ABC Analysis — Revenue-Based Product Classification

In [ ]:
# ── Build product-level revenue summary ──
product_revenue = (order_items
    .groupby('product_id')
    .agg(total_units_sold=('quantity','sum'),
         total_revenue=('line_total','sum'))
    .reset_index()
    .merge(products[['product_id','product_name','category','price',
                      'mrp','margin_percentage','shelf_life_days',
                      'min_stock_level','max_stock_level','is_perishable']],
           on='product_id', how='left')
)

# Sort by revenue descending
product_revenue = product_revenue.sort_values('total_revenue', ascending=False).reset_index(drop=True)

# Cumulative revenue %
product_revenue['cumulative_revenue'] = product_revenue['total_revenue'].cumsum()
total_rev = product_revenue['total_revenue'].sum()
product_revenue['cumulative_pct'] = (product_revenue['cumulative_revenue'] / total_rev * 100).round(2)

# ABC Classification
def assign_abc(cum_pct):
    if cum_pct <= 70:  return 'A'
    elif cum_pct <= 90: return 'B'
    else:               return 'C'

product_revenue['abc_class'] = product_revenue['cumulative_pct'].apply(assign_abc)

# Summary
abc_summary = product_revenue.groupby('abc_class').agg(
    product_count=('product_id','count'),
    total_revenue=('total_revenue','sum'),
    total_units=('total_units_sold','sum')
).reset_index()
abc_summary['revenue_pct'] = (abc_summary['total_revenue'] / total_rev * 100).round(1)
abc_summary['product_pct'] = (abc_summary['product_count'] / len(product_revenue) * 100).round(1)

print('\n📊 ABC Analysis Summary')
print('='*65)
print(f'{"Class":<8} {"Products":>10} {"Prod%":>8} {"Revenue":>14} {"Rev%":>8}')
print('-'*65)
for _, row in abc_summary.iterrows():
    flag = '🔴' if row.abc_class=='A' else '🟡' if row.abc_class=='B' else '🟢'
    print(f"{flag} {row.abc_class:<6} {row.product_count:>10,} {row.product_pct:>7.1f}% "
          f"₹{row.total_revenue:>12,.0f} {row.revenue_pct:>7.1f}%")
print('='*65)

In [ ]:
# ── ABC Pareto Chart ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Pareto Curve
color_map = {'A': '#e74c3c', 'B': '#f39c12', 'C': '#2ecc71'}
bar_colors = [color_map[c] for c in product_revenue['abc_class']]
ax1.bar(range(len(product_revenue)), product_revenue['total_revenue'],
        color=bar_colors, alpha=0.85, width=1.0)
ax1b = ax1.twinx()
ax1b.plot(range(len(product_revenue)), product_revenue['cumulative_pct'],
          color='navy', linewidth=2.5, label='Cumulative %')
ax1b.axhline(70, color='red',    linestyle='--', alpha=0.7, label='70% (A cutoff)')
ax1b.axhline(90, color='orange', linestyle='--', alpha=0.7, label='90% (B cutoff)')
ax1b.set_ylabel('Cumulative Revenue %', color='navy')
ax1b.legend(loc='center right', fontsize=9)
ax1.set_title('📊 ABC Pareto Analysis — Product Revenue', fontsize=14, fontweight='bold')
ax1.set_xlabel('Products (sorted by revenue)')
ax1.set_ylabel('Revenue (₹)')
patches = [mpatches.Patch(color=v, label=f'Class {k}') for k,v in color_map.items()]
ax1.legend(handles=patches, loc='upper right')

# ABC Donut by revenue
ax2.pie(abc_summary['total_revenue'],
        labels=[f"Class {c}\n({p:.0f}% revenue)" for c,p in zip(abc_summary['abc_class'], abc_summary['revenue_pct'])],
        colors=['#e74c3c','#f39c12','#2ecc71'],
        autopct='%1.1f%%', startangle=90,
        wedgeprops={'width':0.5})
ax2.set_title('🍩 ABC Revenue Share', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{CHARTS}/13_abc_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 13_abc_analysis.png')

## 2️⃣ EOQ — Economic Order Quantity

In [ ]:
# ── EOQ Formula: sqrt(2 * D * S / H) ──
# D = Annual Demand (units)
# S = Ordering Cost per order (assumed ₹50 per order for quick-commerce)
# H = Holding Cost per unit per year (assumed 20% of price)

ORDERING_COST = 50     # ₹ per order
HOLDING_RATE  = 0.20   # 20% of unit price per year

# Calculate annual demand from order_items
date_range_days = (orders['order_date'].max() - orders['order_date'].min()).days
date_range_days = max(date_range_days, 1)

product_demand = (order_items
    .groupby('product_id')['quantity'].sum()
    .reset_index()
    .rename(columns={'quantity': 'total_demand'})
)
product_demand['annual_demand'] = (product_demand['total_demand'] * 365 / date_range_days).round(0)

# Merge with products for price
eoq_df = product_demand.merge(
    products[['product_id','product_name','category','price',
              'min_stock_level','max_stock_level','shelf_life_days','abc_class'
              if 'abc_class' in products.columns else 'product_id']],
    on='product_id', how='left'
)

# Merge ABC class
eoq_df = eoq_df.merge(
    product_revenue[['product_id','abc_class']], on='product_id', how='left'
)

# EOQ Calculation
eoq_df['holding_cost'] = eoq_df['price'] * HOLDING_RATE
eoq_df['eoq'] = np.sqrt(
    (2 * eoq_df['annual_demand'] * ORDERING_COST) / eoq_df['holding_cost'].replace(0, np.nan)
).round(0)

# Orders per year
eoq_df['orders_per_year'] = (eoq_df['annual_demand'] / eoq_df['eoq'].replace(0, np.nan)).round(1)

# Total annual cost (ordering + holding)
eoq_df['total_inventory_cost'] = (
    (eoq_df['annual_demand'] / eoq_df['eoq'].replace(0,np.nan)) * ORDERING_COST +
    (eoq_df['eoq'] / 2) * eoq_df['holding_cost']
).round(2)

print('\n📦 EOQ Results (Top 15 by Annual Demand):')
display_cols = ['product_name','category','abc_class','annual_demand','price','eoq','orders_per_year','total_inventory_cost']
display_cols = [c for c in display_cols if c in eoq_df.columns]
print(eoq_df[display_cols].sort_values('annual_demand', ascending=False).head(15).to_string(index=False))

## 3️⃣ Reorder Point (ROP) & Safety Stock

In [ ]:
# ── Reorder Point = (Avg Daily Demand × Lead Time) + Safety Stock ──
# Lead time = 1 day (quick-commerce assumption)
# Safety Stock = Z × σ_demand × sqrt(Lead Time)
# Z = 1.65 (95% service level)

LEAD_TIME = 1     # 1 day for quick-commerce
Z_SCORE   = 1.65  # 95% service level

# Daily demand stats per product
daily_demand = (
    order_items
    .merge(orders[['order_id','order_date']], on='order_id')
    .groupby(['product_id', 'order_date'])['quantity']
    .sum()
    .reset_index()
    .groupby('product_id')['quantity']
    .agg(avg_daily_demand='mean', std_daily_demand='std')
    .reset_index()
    .fillna(0)
)

rop_df = daily_demand.merge(
    products[['product_id','product_name','category','min_stock_level','max_stock_level','shelf_life_days']],
    on='product_id', how='left'
).merge(
    product_revenue[['product_id','abc_class']], on='product_id', how='left'
)

# Safety Stock
rop_df['safety_stock'] = (Z_SCORE * rop_df['std_daily_demand'] * np.sqrt(LEAD_TIME)).round(0)

# Reorder Point
rop_df['reorder_point'] = (rop_df['avg_daily_demand'] * LEAD_TIME + rop_df['safety_stock']).round(0)

# Stock status vs reorder point
rop_df['stock_status'] = rop_df.apply(
    lambda r: '🔴 Below ROP — Restock NOW' if r['min_stock_level'] <= r['reorder_point']
              else '🟢 Stock OK', axis=1
)

# How many need restocking?
needs_restock = (rop_df['min_stock_level'] <= rop_df['reorder_point']).sum()
print(f'\n🔴 Products that need IMMEDIATE restocking: {needs_restock} / {len(rop_df)}')
print(f'\n📋 Reorder Point Analysis (Top 15 at-risk):')
at_risk = rop_df[rop_df['stock_status'].str.contains('Below')].sort_values('reorder_point', ascending=False)
cols = ['product_name','category','abc_class','avg_daily_demand','safety_stock','reorder_point','min_stock_level','stock_status']
cols = [c for c in cols if c in at_risk.columns]
print(at_risk[cols].head(15).to_string(index=False))

In [ ]:
# ── Visualize ROP vs Current Min Stock ──
top_risk = rop_df.sort_values('reorder_point', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(14, 8))
x = range(len(top_risk))
w = 0.35

bars1 = ax.bar([i - w/2 for i in x], top_risk['reorder_point'],
               w, label='🎯 Reorder Point', color='#e74c3c', alpha=0.85)
bars2 = ax.bar([i + w/2 for i in x], top_risk['min_stock_level'],
               w, label='📦 Min Stock Level', color='#3498db', alpha=0.85)

ax.set_xticks(list(x))
ax.set_xticklabels(top_risk['product_name'], rotation=45, ha='right', fontsize=9)
ax.set_title('🎯 Reorder Point vs Min Stock Level (Top 15 At-Risk Products)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Units')
ax.legend(fontsize=11)

# Highlight where min_stock < rop (danger zone)
for i, (rop_val, min_val) in enumerate(zip(top_risk['reorder_point'], top_risk['min_stock_level'])):
    if min_val < rop_val:
        ax.axvspan(i - 0.5, i + 0.5, alpha=0.08, color='red')

plt.tight_layout()
plt.savefig(f'{CHARTS}/14_reorder_point_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 14_reorder_point_analysis.png')

## 4️⃣ Damaged Stock & Wastage Analysis

In [ ]:
# ── Products with highest financial loss from damage ──
damage_by_product = (
    inventory
    .groupby('product_id')
    .agg(total_received=('stock_received','sum'),
         total_damaged=('damaged_stock','sum'),
         avg_damage_rate=('damage_rate','mean'))
    .reset_index()
    .merge(products[['product_id','product_name','category','price','shelf_life_days','is_perishable']],
           on='product_id', how='left')
    .merge(product_revenue[['product_id','abc_class']], on='product_id', how='left')
)

damage_by_product['financial_loss'] = (damage_by_product['total_damaged'] * damage_by_product['price']).round(2)
damage_by_product = damage_by_product.sort_values('financial_loss', ascending=False)

print('💥 Top 15 Products by Financial Loss from Damage:')
print(damage_by_product[['product_name','category','is_perishable','abc_class',
                          'total_damaged','avg_damage_rate','financial_loss']].head(15).to_string(index=False))

total_loss = damage_by_product['financial_loss'].sum()
print(f'\n💸 Total Financial Loss from Damaged Stock: ₹{total_loss:,.2f}')

In [ ]:
# ── Damage Heatmap: Category vs Month ──
inv_merged = inventory.merge(
    products[['product_id','category']], on='product_id', how='left'
)
inv_merged['month_name'] = inv_merged['date'].dt.strftime('%b')
inv_merged['month_num']  = inv_merged['date'].dt.month

heatmap_data = (
    inv_merged.groupby(['category','month_name','month_num'])
    ['damage_rate'].mean()
    .reset_index()
    .sort_values('month_num')
    .pivot(index='category', columns='month_name', values='damage_rate')
)
# Reorder columns by month
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
heatmap_data = heatmap_data.reindex(columns=[m for m in month_order if m in heatmap_data.columns])

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Damage Rate %'})
ax.set_title('🔥 Damage Rate Heatmap — Category vs Month (%)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Month')
ax.set_ylabel('Product Category')
plt.tight_layout()
plt.savefig(f'{CHARTS}/15_damage_heatmap_category_month.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: 15_damage_heatmap_category_month.png')

## 5️⃣ Export Inventory Optimization Report

In [ ]:
# ── Full Inventory Master Report ──
inventory_report = eoq_df.merge(
    rop_df[['product_id','avg_daily_demand','std_daily_demand','safety_stock','reorder_point','stock_status']],
    on='product_id', how='left'
).merge(
    damage_by_product[['product_id','total_damaged','avg_damage_rate','financial_loss']],
    on='product_id', how='left'
)

report_path = f'{REPORTS}/inventory_optimization_report.csv'
inventory_report.to_csv(report_path, index=False)

# Save ABC classified products
abc_path = f'{REPORTS}/abc_classification.csv'
product_revenue.to_csv(abc_path, index=False)

print('✅ Reports saved:')
print(f'   📄 {report_path}')
print(f'   📄 {abc_path}')
print()
print('='*60)
print('     📦 PHASE 4 — INVENTORY OPTIMIZATION SUMMARY')
print('='*60)
print(f'  ABC-A Products (top 70% revenue)  : {(product_revenue.abc_class=="A").sum()}')
print(f'  ABC-B Products (next 20% revenue)  : {(product_revenue.abc_class=="B").sum()}')
print(f'  ABC-C Products (bottom 10% revenue): {(product_revenue.abc_class=="C").sum()}')
print(f'  Products Needing Restock Now        : {needs_restock}')
print(f'  Total Financial Loss (Damage)       : ₹{total_loss:,.0f}')
print(f'  Avg EOQ across all products         : {eoq_df["eoq"].mean():.0f} units')
print('='*60)
print('  ✅ Phase 4 Complete! Ready for Phase 5 — ML Forecasting')
print('='*60)